In [ ]:
import os
import sys
import warnings
import time
import json
from natsort import natsorted
import numpy as np
import xarray as xr
import pandas as pd
import math 
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.pyplot import figure
from matplotlib.patches import Patch, Rectangle
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import matplotlib.colors as mcolors
import seaborn as sns
import cmasher as cmr
from pathlib import PureWindowsPath, PurePosixPath
import pickle
from scipy.signal import find_peaks
import scipy.stats as stats

sys.path.append('../utils') 
from utils_plot import (
    set_pub_style, get_asterisks, save_metadata_json, 
    lighten_color, add_stat_annotation_two_sided, add_significance_bar)

#General parameters
dir_stat = r'../../data/09.Anatomy_behavior'
bin_width = 200  # ms
fs = int(1000/bin_width)

colors_anatomy = ['#A6761D', '#845ec2', '#97cebf'] 
colors_beh_i = ['#4091cf', '#e1703c'] # Blue, red
colors_beh_e = ['#4091cf', '#8cba54'] # Blue, green
#plot
dir_output = r'../output_figures'
os.makedirs(dir_output, exist_ok=True)
dir_fig = 'Fig1'
dpath_plot = os.path.join(dir_output, dir_fig)
if not os.path.exists(dpath_plot):
    os.makedirs(dpath_plot)   

## 1 Anatomy-- Characterization

In [ ]:
df_stat = pd.read_csv(os.path.join(dir_stat, "01_1.Anatomy_characterization.csv")) 

width_mm = 70  
height_mm = 30  
plot_combined_proportions(df_stat, width_mm, height_mm, colors_anatomy, dpath_plot, '01_1.Anatomy_characterization')

In [ ]:
def plot_combined_proportions(df_stat, width_mm, height_mm, colors, output_path, title):
    set_pub_style() 
    # since Plot 1 has 5 groups and Plot 2 has 2 groups.
    fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, 
                                   figsize=(width_mm / 25.4, height_mm / 25.4),
                                   gridspec_kw={'width_ratios': [5, 2]}, layout='constrained')
    
    regions = ['Dorsal', 'Intermediate', 'Ventral']
    group_spacing = 1.2 
    offsets = [-0.3, 0, 0.3]
    box_width = 0.2
    
    metadata = {"Figure_Title": title, "Layer_Data": {}, "Overlap_Data": {}}        
    # ==========================================
    # PANEL 1: Layer Proportions (ax1)
    # ==========================================
    layers = ['EC2', 'EC3', 'EC5a', 'EC5b', 'EC6']        
    layers_name = ['L2', 'L3', 'L5a', 'L5b', 'L6'] 
    for i, layer in enumerate(layers):
        metadata["Layer_Data"][layer] = {}         
        base_x = i * group_spacing        
        for j, region in enumerate(regions):
            region_df = df_stat[df_stat['Region'] == region]
            data = region_df[layer].dropna().values * 100
            n_mice = len(data)
            
            metadata["Layer_Data"][layer][region] = {
                "N_mice": n_mice,
                "Mean": float(np.mean(data)) if n_mice > 0 else 0,
                "Median": float(np.median(data)) if n_mice > 0 else 0,
                "SEM": float(stats.sem(data)) if n_mice > 0 else 0 }
            
            if n_mice == 0: continue
            
            x_pos = base_x + offsets[j]
            color = colors[j]   
            face_color_rgba = mcolors.to_rgba(color, alpha=0.3)                        
            ax1.boxplot(data, positions=[x_pos], widths=box_width, patch_artist=True, showfliers=False,
                        boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5), 
                        whiskerprops=dict(color=color, linewidth=0.8),
                        capprops=dict(color='none', linewidth=0),
                        medianprops=dict(color=color, linewidth=1.0, zorder=4), zorder=2)

            x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=n_mice)
            ax1.scatter(x_jitter, data, color=color, edgecolor='white', linewidth=0.25, s=2.0, zorder=3, alpha=1.0)

    # Ax1 Aesthetics
    ax1.set_xticks([i * group_spacing for i in range(len(layers_name))])
    ax1.set_xticklabels(layers_name, fontsize=7)
    ax1.set_ylabel("EYFP+ cells (%)", labelpad=0.1)
    
    for tick_label in ax1.get_xticklabels():
        if tick_label.get_text() == 'EC5b':
            tick_label.set_fontweight('bold')
            
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(20))   
    ax1.grid(axis='y', color='gray', linestyle='--', linewidth=0.5, alpha=0.4, zorder=0)  
    bottom, top = ax1.get_ylim()
    ax1.set_ylim(max(0, bottom), top)
    
    legend_handles = []
    for j, region in enumerate(regions):
        color = colors[j]
        face_color_rgba = mcolors.to_rgba(color, alpha=0.3)
        legend_handles.append(mpatches.Patch(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5, label=region))
    ax1.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.05, 0.98), frameon=False, 
           fontsize=5, handletextpad=0.5, borderpad=0, handlelength=1, handleheight=0.7)
    
    # ==========================================
    # PANEL 2: Overlap Proportions (ax2)
    # ==========================================
    metrics = ['Overlap_to_Pcp4', 'Overlap_to_EYFP']
    metric_labels = ['Pcp4+', 'EYFP+']
    
    for i, metric in enumerate(metrics):
        metadata["Overlap_Data"][metric] = {}
        base_x = i * group_spacing
        
        for j, region in enumerate(regions):
            region_df = df_stat[df_stat['Region'] == region]
            data = region_df[metric].dropna().values * 100
            n_mice = len(data)
            
            metadata["Overlap_Data"][metric][region] = {
                "N_mice": n_mice,
                "Mean": float(np.mean(data)) if n_mice > 0 else 0,
                "Median": float(np.median(data)) if n_mice > 0 else 0,
                "SEM": float(stats.sem(data)) if n_mice > 0 else 0 }
            
            if n_mice == 0: continue
            
            x_pos = base_x + offsets[j]
            color = colors[j]   
            face_color_rgba = mcolors.to_rgba(color, alpha=0.3)            
            
            ax2.boxplot(data, positions=[x_pos], widths=box_width, patch_artist=True, showfliers=False,
                        boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5), 
                        whiskerprops=dict(color=color, linewidth=0.8),
                        capprops=dict(color='none', linewidth=0),
                        medianprops=dict(color=color, linewidth=1.0, zorder=4), zorder=2)
            
            x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=n_mice)
            ax2.scatter(x_jitter, data, color=color, edgecolor='white', linewidth=0.25, s=2.0, zorder=3, alpha=1.0)            
            
    # Ax2 Aesthetics
    ax2.set_xticks([i * group_spacing for i in range(len(metrics))])
    ax2.set_xticklabels(metric_labels, fontsize=6)
    ax2.set_ylabel("Double-positive cells (%)", labelpad=0.1)
    
    ax2.yaxis.set_major_locator(ticker.MultipleLocator(20))    
    ax2.grid(axis='y', color='gray', linestyle='--', linewidth=0.5, alpha=0.4, zorder=0) 
    ax2.set_ylim(0, 100)
   
    base_path = os.path.join(output_path, title.replace(' ', '_'))  
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()  
    
    save_metadata_json(metadata, output_path, title)

## 2 Anatomy -- Retrograde

In [ ]:
# EC5b
df_stat = pd.read_csv(os.path.join(dir_stat, "01_2.Anatomy_retrograde_EC5b.csv")) 
width_mm = 55  
height_mm = 45    
# 1. Define the anatomical mapping (1-indexed logic applied later)
high_brains = { 'HPC': [3, 6], 'Cortex': [7,15], 'TH': [16,16],'BF':[17,18],  'Others':[19,20]} #Column of brain regions
plot_cell_retro_proportions(df_stat, 'EC5b', high_brains, width_mm, height_mm, dpath_plot, '02_1.Anatomy_retro_EC5b') 

# EC5a
df_stat = pd.read_csv(os.path.join(dir_stat, "01_3.Anatomy_retrograde_EC5a.csv")) 

width_mm = 40  
height_mm = 30 
high_brains = { 'HPC': [3, 6], 'L5b': [7,7]} #Column of brain regions
plot_cell_retro_proportions(df_stat, 'EC5a', high_brains, width_mm, height_mm, dpath_plot, 'sup_02_1.Anatomy_retro_EC5a')

In [ ]:
def save_metadata_json(metadata_dict, output_path, title):
    os.makedirs(output_path, exist_ok=True)
    file_path = os.path.join(output_path, f"{title.replace(' ', '_')}_caption_data.json")
    with open(file_path, 'w') as f:
        json.dump(metadata_dict, f, indent=4)

def plot_cell_retro_proportions(df_stat, flag, high_brains, width_mm, height_mm, output_path, title):
    set_pub_style()   
    # 1. Create the figure AND the main bar axis using subplots so Constrained Layout works!
    fig, ax_bar = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')     
    # 2. Create the floating pie chart as an INSET of the bar chart.
    # [left, bottom, width, height] - These are now fractions of ax_bar, not the whole figure!
    if flag == 'EC5b':
        ax_pie = ax_bar.inset_axes([0.15, 0, 0.5, 0.5])
    if flag == 'EC5a':
        ax_pie = ax_bar.inset_axes([0.4, 0, 0.40, 0.50])
        
    metadata = {"Figure_Title": title, "Detailed_Regions": {}, "High_Level_Regions": {}}   
    # ==========================================
    # PART A: THE DETAILED BAR PLOT
    # ==========================================
    bar_color = '#c44e52' # Muted Red
    
    detailed_names = []
    detailed_means = []
    detailed_sems = []
    
    for region, bounds in high_brains.items():
        start_col = bounds[0]
        end_col = bounds[1]
        
        for col_idx in range(start_col, end_col + 1):
            col_name = df_stat.columns[col_idx]
            
            data = df_stat[col_name].dropna().values * 100
            
            n_mice = len(data)
            mean_val = float(np.mean(data)) if n_mice > 0 else 0
            sem_val = float(stats.sem(data)) if n_mice > 0 else 0
            
            detailed_names.append(col_name)
            detailed_means.append(mean_val)
            detailed_sems.append(sem_val)
            
            metadata["Detailed_Regions"][col_name] = {
                "N_mice": n_mice, "Mean": mean_val, "SEM": sem_val }
            
    y_pos = np.arange(len(detailed_names))    
    bar_height = 0.35 if flag == 'EC5a' else 0.7
    ax_bar.barh(y_pos, detailed_means, xerr=detailed_sems, height=bar_height, 
                color=bar_color, edgecolor='none', alpha=1.0, 
                error_kw=dict(lw=0.75, ecolor='black', capsize=0))
    
    for idx, col_name in enumerate(detailed_names):
        if col_name=='EC5b':
            detailed_names[idx] = 'L5b'
        
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(detailed_names, fontsize=6)# 
    ax_bar.invert_yaxis() 
    ax_bar.set_xlabel("Retrogradely labeled cells (%)", labelpad=0.1)
    
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.grid(axis='x', color='gray', linestyle='--', linewidth=0.5, alpha=0.4, zorder=0)

    # ==========================================
    # PART B: THE HIGH-LEVEL PIE CHART
    # ==========================================
    pie_labels = list(high_brains.keys())
    pie_means = []
    pie_colors = ['#48245c', '#8172b2', '#c44e52', '#e6a5a6', '#c5cbd3']
    
    for region, bounds in high_brains.items():
        start_col = bounds[0]
        end_col = bounds[1]        
        mouse_sums = df_stat.iloc[:, start_col:end_col + 1].sum(axis=1).dropna().values * 100
        mean_sum = float(np.mean(mouse_sums))
        pie_means.append(mean_sum)
        
        metadata["High_Level_Regions"][region] = {
            "N_mice": len(mouse_sums),
            "Mean_Total_Percentage": mean_sum,
            "SEM_Total_Percentage": float(stats.sem(mouse_sums))
        }
    ax_pie.pie(pie_means, colors=pie_colors, 
               wedgeprops=dict(edgecolor='white', linewidth=1.0))
               
    ax_pie.legend(pie_labels, loc='center left', bbox_to_anchor=(1.05, 0.5), 
              frameon=False, fontsize=7, ncol=1)

    # ==========================================
    # EXPORTING
    # ==========================================
    base_path = os.path.join(output_path, title.replace(' ', '_'))    
    # It will pad the Bar Chart and ignore the floating pie.
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")   
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close() 
    
    save_metadata_json(metadata, output_path, title)

## 3 Anatomy -- Anterograde

In [ ]:
df_stat = pd.read_csv(os.path.join(dir_stat, "01_4.Anatomy_anterograde_EC5b.csv")) 
width_mm = 60  
height_mm = 30  
plot_antero_intensity_proportions(df_stat, width_mm, height_mm, colors_anatomy, dpath_plot, '03_1.Anatomy_anterograde')

# SC to PC proportion in supplermentary
width_mm = 40  
height_mm = 30
plot_sc_to_pc_chance_significance(df_stat, width_mm, height_mm, colors_anatomy, dpath_plot, 'sup_03_1.Anatomy_anterograde_SC_to_PC')

In [ ]:
def plot_antero_intensity_proportions(df_stat, width_mm, height_mm, colors, output_path, title):
    set_pub_style()    
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    regions = ['Dorsal', 'Medial', 'Ventral']
    layers = ['EC1', 'EC2', 'EC3', 'EC5a', 'EC5b', 'EC6']     
    layers_name = ['L1', 'L2', 'L3', 'L5a', 'L5b', 'L6'] 
    # Increase the multiplier to create physical space between layer groups
    group_spacing = 1.5
    offsets = [-0.3, 0, 0.3]
    bar_width = 0.25
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}    
    for i, layer in enumerate(layers):
        metadata["Data_Summary"][layer] = {}        
        base_x = i * group_spacing        
        for j, region in enumerate(regions):
            region_df = df_stat[df_stat['Region'] == region]
            data = region_df[layer].dropna().values
            n_mice = len(data)
            
            metadata["Data_Summary"][layer][region] = {
                "N_mice": n_mice,
                "Mean": float(np.mean(data)) if n_mice > 0 else 0,
                "SEM": float(stats.sem(data)) if n_mice > 0 else 0 
            }
            
            if n_mice == 0: continue            
            x_pos = base_x + offsets[j]
            color = colors[j]            
            
            mean_val = np.mean(data)
            sem_val = stats.sem(data)            
            # 1. Bar Plot (Clean, borderless style for modern look)
            ax.bar(x_pos, mean_val, yerr=sem_val, width=bar_width, 
                   color=color, alpha=1.0, edgecolor='none', capsize=0,
                   error_kw=dict(lw=0.75, ecolor='black'), zorder=1)
            
            # 2. Scatter Raw Data (Microscopic points so they don't visually overwhelm the bar)
            # 1. Create perfectly even, strictly non-overlapping spacing
            jitter_offsets = np.linspace(-0.06, 0.06, n_mice)
            np.random.shuffle(jitter_offsets)
            x_jitter = x_pos + jitter_offsets  
            #x_jitter = x_pos + np.random.uniform(-0.06, 0.06, size=n_mice)
            ax.scatter(x_jitter, data, color=color, edgecolor='white', linewidth=0.25, 
                       s=2.5, zorder=2, alpha=1.0)

    # Aesthetics & Formatting
    ax.set_xticks([i * group_spacing for i in range(len(layers_name))])
    ax.set_xticklabels(layers_name, fontsize=7)
    ax.set_ylabel("Projection intensity\n(Normalized to EC5b)", labelpad=0.1)
    # Force Y-axis to step by exactly 10
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1.0))  
    ax.axhline(y=1.0,color='gray',linestyle='--',linewidth=0.5,alpha=0.4,zorder=0)
    # Bold the EC5b label
    for tick_label in ax.get_xticklabels():
        if tick_label.get_text() == 'L5b':
            tick_label.set_fontweight('bold')    
    # Save Outputs  
    base_path = os.path.join(output_path, title.replace(' ', '_'))    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()       
    save_metadata_json(metadata, output_path, title)

def plot_sc_to_pc_chance_significance(df_stat, width_mm, height_mm, colors, output_path, title):
    """
    Plots a bar/scatter plot for a single metric across regions, 
    testing significance against a chance level of 1.0.
    """
    set_pub_style()     
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    layer = 'SC_to_PC'
    regions = ['Dorsal', 'Medial', 'Ventral']
    display_regions = ['Dorsal', 'Inter-\nmediate', 'Ventral'] # For the X-axis labels
    
    chance_level = 1.0
    bar_width = 0.5 # Wider bars since they are now the primary categories
    
    metadata = {"Figure_Title": title, "Statistical_Test": "1-Sample T-Test against chance (mu=1.0)", "Data_Summary": {}}    
    
    # Pre-calculate global max for scaling stars and Y-axis
    global_max = 0
    
    for i, region in enumerate(regions):
        region_df = df_stat[df_stat['Region'] == region]
        data = region_df[layer].dropna().values
        
        if len(data) > 0:
            local_max = np.max(data)
            global_max = max(global_max, local_max)
            
    for i, region in enumerate(regions):
        region_df = df_stat[df_stat['Region'] == region]
        data = region_df[layer].dropna().values
        n_mice = len(data)
        
        if n_mice == 0: continue
        
        x_pos = i
        color = colors[i]            
        
        mean_val = np.mean(data)
        sem_val = stats.sem(data)
        
        # --- Significance Testing (1-Sample T-Test against 1.0) ---
        p_val = 'ns'
        t_stat = 0
        star = ''
        if n_mice >= 3: # Need at least 3 samples for a robust t-test
            #t_stat, p_val = stats.wilcoxon(data - chance_level, alternative='two-sided')
            t_stat, p_val = stats.ttest_1samp(data, popmean=chance_level, alternative='two-sided')
            star = get_asterisks(p_val)
        
        # Save to JSON
        metadata["Data_Summary"][region] = {
            "N_mice": n_mice,
            "Mean": float(mean_val),
            "SEM": float(sem_val),
            "t_statistic": float(t_stat),
            "p_value": float(p_val) if p_val != 'ns' else 'ns',
        }
        
        # 1. Bar Plot
        ax.bar(x_pos, mean_val, yerr=sem_val, width=bar_width, 
               color=color, alpha=1.0, edgecolor='none', capsize=0,
               error_kw=dict(lw=0.75, ecolor='black'), zorder=1)
        
        # 2. Scatter Raw Data
        jitter_offsets = np.linspace(-0.06, 0.06, n_mice)
        np.random.shuffle(jitter_offsets)
        x_jitter = x_pos + jitter_offsets  
        ax.scatter(x_jitter, data, color=color, edgecolor='white', linewidth=0.25, 
                   s=4, zorder=2, alpha=1.0)
                   
        # 3. Plot Significance Stars
        if star and star != 'ns':
            # Place the star safely above the highest point (either error bar or max scatter dot)
            y_top = max(mean_val + sem_val, np.max(data))
            ax.text(x_pos, y_top + (global_max * 0.05), star, 
                    ha='center', va='center', color='k', fontsize=7)

    # --- Chance Level Reference Line ---
    ax.axhline(y=chance_level, color='#444444', linestyle='--', linewidth=1.0, alpha=0.8, zorder=0)
    
    # --- Aesthetics & Formatting ---
    ax.set_xticks(range(len(regions)))
    ax.set_xticklabels(display_regions, fontsize=7)
    ax.set_ylabel(f"Axon density\n(SC / PC)")
    
    # Dynamically set Y-axis
    ax.set_ylim(0, global_max * 1.05)
    
    # Force clean, optimal Y-axis steps (e.g., 0, 1, 2, 3)
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5]))  

    base_path = os.path.join(output_path, title.replace(' ', '_'))
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   #bbox_inches='tight', pad_inches=0.02
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()       
    save_metadata_json(metadata, output_path, title)
